Cuaderno 2: Entrenamiento del Modelo

En esta primera celda, importamos las librerías necesarias para el preprocesamiento matemático, la creación del modelo predictivo y la exportación del mismo. También configuramos las rutas absolutas para asegurar que el cuaderno encuentre el archivo .parquet que generamos en el paso anterior y sepa dónde guardar el modelo final.

In [6]:
import os
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

print("Importando librerías de Machine Learning (Pandas, XGBoost, Scikit-Learn)...")
print("Librerías importadas correctamente.")

# Configuración de directorios
BASE_DIR = os.path.abspath('')
DATA_DIR = os.path.join(BASE_DIR, "data", "raw")
MODELS_DIR = os.path.join(BASE_DIR, "models")

# Creación de carpeta models si no existe
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Directorio de modelos configurado en: {MODELS_DIR}")

SYMBOL = "EURUSD"
RUTA_DATOS = os.path.join(DATA_DIR, f"{SYMBOL}_M5.parquet")
RUTA_MODELO = os.path.join(MODELS_DIR, f"{SYMBOL}_M5_scalper.joblib")

print(f"Símbolo a modelar: {SYMBOL}")
print(f"Ruta de lectura de datos: {RUTA_DATOS}")

ModuleNotFoundError: No module named 'pandas'

Ahora procedemos a cargar los datos procesados. Una vez cargados, filtramos el histórico para trabajar con el periodo establecido (2023-2025). Luego, dividimos los datos en dos bloques: uno para entrenar al modelo (In-Sample) y otro para validar su comportamiento con datos que nunca ha visto (Out-of-Sample).

In [7]:
print("Iniciando carga de datos Parquet...")
if not os.path.exists(RUTA_DATOS):
    print("ERROR CRÍTICO: No se encontró el archivo Parquet. Ejecuta el Cuaderno 1 primero.")
else:
    df_feat = pd.read_parquet(RUTA_DATOS)
    print(f"Datos cargados exitosamente. Filas totales disponibles: {len(df_feat)}")

    print("Aplicando recorte temporal global: 01-01-2023 al 31-12-2025...")
    df_feat = df_feat.loc['2023-01-01':'2025-12-31']
    print(f"Filas resultantes tras el recorte temporal: {len(df_feat)}")

    print("Dividiendo dataset: In-Sample (Entrenamiento) y Out-of-Sample (Validación)...")
    df_is = df_feat.loc['2023-01-01':'2024-12-31']
    df_oos = df_feat.loc['2025-01-01':'2025-12-31']

    print(f"  -> Conjunto de Entrenamiento (IS): {len(df_is)} filas.")
    print(f"  -> Conjunto de Validación (OOS): {len(df_oos)} filas.")

Iniciando carga de datos Parquet...


NameError: name 'RUTA_DATOS' is not defined

En esta fase, extraemos las características (features) y la variable objetivo (target) y las convertimos en matrices numéricas que el algoritmo puede procesar. Como en el trading suele haber un desbalance (hay menos operaciones exitosas que velas en el gráfico), calculamos un peso matemático (scale_pos_weight) para equilibrar el aprendizaje del modelo.

In [8]:
print("Seleccionando características (features) para el entrenamiento...")
cols_features = [
    'z_score', 'dist_ema288', 'vol_spike', 
    'lower_wick_ratio', 'upper_wick_ratio', 'body_ratio'
]
print(f"Características utilizadas: {cols_features}")

# Separación de X (features) e y (target)
X_train = df_is[cols_features].values
y_train = df_is['target'].values
X_test = df_oos[cols_features].values
y_test = df_oos['target'].values
print("Matrices de entrenamiento y validación generadas.")

print("Calculando balanceo de clases...")
sum_y = sum(y_train)
len_y = len(y_train)
print(f"  -> Total muestras entrenamiento: {len_y}")
print(f"  -> Casos Positivos (Target=1): {sum_y}")
print(f"  -> Casos Negativos (Target=0): {len_y - sum_y}")

scale_weight = (len_y - sum_y) / sum_y if sum_y > 0 else 1
scale_weight = min(scale_weight, 15)
print(f"Peso asignado a la clase positiva (scale_pos_weight): {scale_weight:.2f}")

Seleccionando características (features) para el entrenamiento...
Características utilizadas: ['z_score', 'dist_ema288', 'vol_spike', 'lower_wick_ratio', 'upper_wick_ratio', 'body_ratio']


NameError: name 'df_is' is not defined

Llegó el momento de construir la arquitectura del modelo. Creamos un Pipeline de Scikit-Learn que primero estandariza los datos (StandardScaler) para que todas las métricas estén en la misma escala, y luego aplica el clasificador XGBoost con los hiperparámetros optimizados. Una vez configurado, procedemos a entrenarlo.

In [5]:
print("Construyendo el Pipeline del modelo (Escalado + XGBoost)...")
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(
        objective='binary:logistic',
        tree_method='hist',
        device='cuda', # Si tienes problemas con CUDA, cambia esto a 'cpu'
        n_estimators=700,
        learning_rate=0.02,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=1.0,
        scale_pos_weight=scale_weight,
        random_state=42
    ))
])

print("Iniciando proceso de entrenamiento (Fit)...")
print("El modelo está aprendiendo patrones, esto puede tomar unos segundos...")
pipeline.fit(X_train, y_train)
print("¡Entrenamiento finalizado con éxito!")

print("Configurando el modelo entrenado para ejecutarse en CPU para la inferencia...")
pipeline.named_steps['xgb'].get_booster().set_param({'device': 'cpu'})

Construyendo el Pipeline del modelo (Escalado + XGBoost)...
Iniciando proceso de entrenamiento (Fit)...
El modelo está aprendiendo patrones, esto puede tomar unos segundos...
¡Entrenamiento finalizado con éxito!
Configurando el modelo entrenado para ejecutarse en CPU para la inferencia...


Para comprobar que el modelo ha aprendido algo coherente antes de llevarlo al Backtest, realizamos una predicción rápida sobre el conjunto de datos Out-of-Sample y mostramos su precisión global. Finalmente, exportamos el Pipeline completo como un archivo .joblib para poder usarlo en el Cuaderno 3 o en producción.

In [7]:
print("Realizando predicción preliminar sobre el conjunto de Validación (OOS)...")
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión global (Accuracy) en OOS: {accuracy * 100:.2f}%")

print("Preparando exportación del modelo...")
joblib.dump(pipeline, RUTA_MODELO)
print(f"¡Modelo guardado exitosamente en: {RUTA_MODELO}!")

print("\n" + "="*50)
print("  FIN DEL CUADERNO 2 - MODELO LISTO PARA BACKTEST  ")
print("="*50)

Realizando predicción preliminar sobre el conjunto de Validación (OOS)...
Precisión global (Accuracy) en OOS: 61.99%
Preparando exportación del modelo...
¡Modelo guardado exitosamente en: c:\Users\juand\OneDrive\Documentos\ECHELONIX\NO SUBIR\vscode\CUADERNOS JUPYTER\models\EURUSD_M5_scalper.joblib!

  FIN DEL CUADERNO 2 - MODELO LISTO PARA BACKTEST  


In [4]:
# Instalar si no lo tienes: %pip install skl2onnx onnxmltools
import onnxmltools
from skl2onnx.common.data_types import FloatTensorType
import os

print("Exportando modelo a formato ONNX para cumplir con el requisito del profesor...")

# Definimos cuántas variables de entrada tiene nuestro modelo (son 6 features)
initial_type = [('float_input', FloatTensorType([None, 6]))]

# Convertimos nuestro Pipeline entrenado a formato ONNX
onnx_model = onnxmltools.convert_sklearn(pipeline, initial_types=initial_type)

# Guardamos el archivo con la extensión que pide el profesor
ruta_onnx = os.path.join(MODELS_DIR, f"{SYMBOL}_M5_scalper.onnx")
onnxmltools.utils.save_model(onnx_model, ruta_onnx)

print(f"¡Modelo optimizado exportado con éxito en formato ONNX: {ruta_onnx}!")

Exportando modelo a formato ONNX para cumplir con el requisito del profesor...


NameError: name 'pipeline' is not defined